# Análise de características usando um Multilayer Perceptron

> **Autores:** Alexandre Maciel e Vinicius de Lima.

Esse notebook calcula o F1-score de um MLP para diferentes tamanhos para as bases geradas por [bases.ipynb](./bases.ipynb), com as caractéristicas extraídas das raças de cachorro Samoieda e Pug e das raças de gato Russian Blue e Birman.

In [1]:
from itertools import product
from typing import Literal

import pandas as pd
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score

Nós temos 4 raças de cachorro no total, portanto 4 classes $(N_o = 4)$, além disso a melhor base foi `PCA_075_HOG_256_32x32`, que possui 83 atributos $(N_i = 83)$. Então, a 
quantidade de neurônios a partir da regra geométrica $\lfloor\sqrt{N_i \cdot N_o}\rfloor$, teremos 18 neurônios na camada escondida.

$$
\lfloor\sqrt{N_i \cdot N_o}\rfloor = 18
$$

Aplicando a mesma regra para a regra da média, teremos:

$$
\lfloor\frac{2}{3} N_i + N_o \rfloor = 59
$$

In [ ]:
DATASET = "PCA_075_HOG_256_32x32.csv"

df = pd.read_csv("bases/" + DATASET)
xs = df.iloc[:, :-1]
ys = df["label"]

xs_train, xs_test, ys_train, ys_test = train_test_split(xs, ys, random_state=42)

activation: Literal["relu"] = "relu"
hidden_layer_sizes = [18, 59]
solvers: list[Literal["adam", "sgd"]] = ["adam", "sgd"]
learning_rates = [0.0001, 0.001, 0.01, 0.1]
number_of_iterations = [1000, 1500, 2000, 2500, 3000]

In [ ]:
results = []

for hl, s, ln, its in product(
    hidden_layer_sizes, solvers, learning_rates, number_of_iterations
):
    clf = MLPClassifier(
        hidden_layer_sizes=hl,
        activation=activation,
        solver=s,
        learning_rate_init=ln,
        max_iter=its,
    )
    _ = clf.fit(xs_train, ys_train)
    f1 = f1_score(ys_test, clf.predict(xs_test), average="weighted")
    results.append(
        {
            "hidden_layer_sizes": hl,
            "solver": s,
            "learning_rate_init": ln,
            "max_iter": its,
            "activation": activation,
            "f1_weighted": f1,
        }
    )


In [18]:
results_df = pd.DataFrame(results)
results_df.to_csv("mlp/results.csv", index=False)
results_df

,hidden_layer_sizes,solver,learning_rate_init,max_iter,activation,f1_weighted
0,18,adam,0.0001,1000,relu,0.669205
1,18,adam,0.0001,1500,relu,0.654819
2,18,adam,0.0001,2000,relu,0.685024
3,18,adam,0.0001,2500,relu,0.679904
4,18,adam,0.0001,3000,relu,0.650000
...,...,...,...,...,...,...
75,59,sgd,0.1000,1000,relu,0.690031
76,59,sgd,0.1000,1500,relu,0.694656
77,59,sgd,0.1000,2000,relu,0.720000
78,59,sgd,0.1000,2500,relu,0.714679


In [8]:
sorted = results_df.nlargest(10, "f1_weighted")
sorted

,hidden_layer_sizes,solver,learning_rate_init,max_iter,activation,f1_weighted
43,59,adam,0.0001,2500,relu,0.764546
59,59,adam,0.1000,3000,relu,0.749624
25,18,sgd,0.0010,1000,relu,0.740000
52,59,adam,0.0100,2000,relu,0.740000
70,59,sgd,0.0100,1000,relu,0.734967
56,59,adam,0.1000,1500,relu,0.730027
60,59,sgd,0.0001,1000,relu,0.729784
71,59,sgd,0.0100,1500,relu,0.724855
18,18,adam,0.1000,2500,relu,0.724690
48,59,adam,0.0010,2500,relu,0.720028


In [9]:
sorted.to_csv("mlp/sorted.csv", index=False)

In [5]:
datasets = [
    "PCA_075_HOG_256_32x32.csv",
    "PCA_075_HOG_128_16x16.csv",
    "PCA_090_HOG_256_32x32.csv",
    "HOG_256_32x32.csv",
    "PCA_090_HOG_128_16x16.csv",
    "PCA_075_HOG_128_32x32.csv",
    "PCA_090_HOG_128_32x32.csv",
    "HOG_128_16x16.csv",
    "HOG_128_32x32.csv",
    "LBP_256_6R.csv",
    "PCA_075_HOG_256_16x16.csv",
    "LBP_256_12R.csv",
]

In [7]:
final_results = []

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for dataset in datasets:
    print("dataset:", dataset)
    
    df = pd.read_csv("bases/" + dataset)
    xs = df.iloc[:, :-1]
    ys = df["label"]
    xs_train, xs_test, ys_train, ys_test = train_test_split(xs, ys, random_state=42)

    row_10fold = {"dataset": dataset, "split": "10-fold CV"}
    row_70_30 = {"dataset": dataset, "split": "70/30"}

    for i, (_, row) in enumerate(results_df.iterrows()):
        print("conf:", i)

        clf = MLPClassifier(
            hidden_layer_sizes=row["hidden_layer_sizes"],
            activation=row["activation"],
            solver=row["solver"],
            learning_rate_init=row["learning_rate_init"],
            max_iter=row["max_iter"],
        )

        # 10-fold CV
        f1_cv = cross_val_score(clf, xs, ys, cv=kf, scoring="f1_weighted").mean()
        row_10fold[f"conf {i}"] = f1_cv

        # Holdout
        _ = clf.fit(xs_train, ys_train)
        f1_holdout = f1_score(ys_test, clf.predict(xs_test), average="weighted")
        row_70_30[f"conf {i}"] = f1_holdout

    final_results.append(row_10fold)
    final_results.append(row_70_30)

/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/vinicius/dev/uf/img-ft-extraction-ml/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10

In [8]:
final_results_df = pd.DataFrame(final_results)
final_results_df.to_csv("mlp/final_results.csv", index=False)
final_results_df

,dataset,split,conf 0,conf 1,conf 2,conf 3,conf 4,conf 5,conf 6,conf 7,conf 8,conf 9
0,PCA_075_HOG_256_32x32.csv,10-fold CV,0.725878,0.719678,0.693762,0.765110,0.727500,0.712291,0.671989,0.728582,0.707478,0.720775
1,PCA_075_HOG_256_32x32.csv,70/30,0.734701,0.684392,0.700030,0.744866,0.664958,0.674634,0.695023,0.700030,0.685024,0.679904
2,PCA_075_HOG_128_16x16.csv,10-fold CV,0.698133,0.675982,0.693053,0.691012,0.704154,0.673156,0.668693,0.705667,0.694192,0.684299
3,PCA_075_HOG_128_16x16.csv,70/30,0.674634,0.659898,0.688912,0.649895,0.700000,0.660000,0.704431,0.700000,0.674959,0.689752
4,PCA_090_HOG_256_32x32.csv,10-fold CV,0.685858,0.707137,0.705527,0.718078,0.726572,0.703299,0.716309,0.716748,0.699411,0.690410
5,PCA_090_HOG_256_32x32.csv,70/30,0.621616,0.698095,0.674634,0.704667,0.704845,0.648309,0.724966,0.689534,0.689752,0.669205
6,HOG_256_32x32.csv,10-fold CV,0.701658,0.478810,0.723681,0.579769,0.701349,0.342840,0.545515,0.712871,0.415538,0.698492
7,HOG_256_32x32.csv,70/30,0.684834,0.518068,0.744713,0.674634,0.689907,0.344503,0.520725,0.670000,0.322282,0.664824
8,PCA_090_HOG_128_16x16.csv,10-fold CV,0.679849,0.701761,0.701474,0.714753,0.701688,0.689899,0.695184,0.689707,0.685526,0.687447
9,PCA_090_HOG_128_16x16.csv,70/30,0.585118,0.590041,0.704667,0.660000,0.684645,0.645027,0.714850,0.619886,0.654611,0.623898


In [9]:
conf_columns = [f"conf {i}" for i in range(12)]
final_results_df.describe()

,conf 0,conf 1,conf 2,conf 3,conf 4,conf 5,conf 6,conf 7,conf 8,conf 9
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.632537,0.628827,0.634757,0.676673,0.677530,0.630465,0.576476,0.674894,0.616743,0.672438
std,0.124739,0.104332,0.138850,0.053486,0.065028,0.106289,0.160300,0.058703,0.124824,0.046454
min,0.344503,0.322282,0.322282,0.579769,0.481701,0.342840,0.317289,0.523731,0.322282,0.584782
25%,0.616182,0.594516,0.643423,0.624632,0.684882,0.605848,0.459685,0.662509,0.592950,0.649940
50%,0.684992,0.661784,0.695928,0.688018,0.701519,0.655000,0.670551,0.694366,0.667691,0.684017
75%,0.706223,0.702488,0.705833,0.715786,0.711255,0.679814,0.706285,0.713840,0.695497,0.702539
max,0.739243,0.733379,0.744713,0.765110,0.727500,0.754871,0.724966,0.733967,0.715021,0.744508
